# 09 - Advanced RAG Techniques

The basic RAG pipeline from notebook 06 works, but there's a lot of room for improvement. This notebook covers three advanced techniques that can significantly boost retrieval quality:

1. **HyDE (Hypothetical Document Embeddings)** — better query embeddings
2. **Multi-Query Retrieval** — broader coverage
3. **Re-Ranking with Cross-Encoders** — more precise final results

Each technique addresses a different weakness in the basic pipeline.

In [ ]:
import os
import sys
import tempfile

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))

from rag_engine.chains.prompts import HYDE_PROMPT
from rag_engine.chunking.strategies import chunk_documents
from rag_engine.llm.provider import LLMProvider
from rag_engine.loaders import load_documents
from rag_engine.vectorstore.chroma_store import add_documents

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "sample")

In [ ]:
# Build knowledge base (same setup as previous notebooks)
md_docs = load_documents(os.path.join(DATA_DIR, "rag_survey.md"))
csv_docs = load_documents(os.path.join(DATA_DIR, "ml_glossary.csv"))
chunks = chunk_documents(md_docs + csv_docs, chunk_size=256, chunk_overlap=30)

temp_dir = tempfile.mkdtemp()
vectorstore = add_documents(chunks, collection_name="nb09", persist_directory=temp_dir)
llm = LLMProvider.get_llm(temperature=0.0)

print(f"Indexed {len(chunks)} chunks")

## Technique 1: HyDE (Hypothetical Document Embeddings)

**Problem:** User queries are often short and don't match the style of the documents in the knowledge base. "What is chunking?" doesn't embed similarly to a paragraph explaining chunking.

**Solution:** Generate a **hypothetical answer** first, then embed *that* for retrieval. The hypothetical answer is closer in style to the actual documents.

```
Query → LLM generates hypothetical answer → Embed hypothetical → Retrieve similar docs
```

In [ ]:
from langchain_core.output_parsers import StrOutputParser

query = "How does chunking affect retrieval performance?"

# Step 1: Generate hypothetical answer
hyde_chain = HYDE_PROMPT | llm | StrOutputParser()
hypothetical_answer = hyde_chain.invoke({"question": query})

print("Original query:")
print(f"  {query}")
print("\nHypothetical answer (generated by LLM):")
print(f"  {hypothetical_answer[:300]}...")

In [ ]:
# Step 2: Compare retrieval results

# Baseline: retrieve with original query
baseline_results = vectorstore.similarity_search(query, k=3)

# HyDE: retrieve with hypothetical answer
hyde_results = vectorstore.similarity_search(hypothetical_answer, k=3)

print("=== Baseline Retrieval (original query) ===")
for i, doc in enumerate(baseline_results, 1):
    print(f"  [{i}] {doc.page_content[:120]}...")

print("\n=== HyDE Retrieval (hypothetical answer) ===")
for i, doc in enumerate(hyde_results, 1):
    print(f"  [{i}] {doc.page_content[:120]}...")

## Technique 2: Multi-Query Retrieval

**Problem:** A single query may not capture all aspects of what the user is asking. "Tell me about RAG evaluation" could mean metrics, frameworks, or methodology.

**Solution:** Use the LLM to generate **multiple query variations**, retrieve for each, then merge the results.

```
Query → LLM generates 3 variations → Retrieve for each → Merge & deduplicate
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

multi_query_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an AI assistant that generates multiple search queries based on a single input query. "
     "Generate 3 different versions of the given question to retrieve relevant documents. "
     "Each version should approach the topic from a different angle. "
     "Output only the 3 questions, one per line, numbered 1-3."),
    ("human", "{question}"),
])

query = "How should I evaluate my RAG system?"

multi_chain = multi_query_prompt | llm | StrOutputParser()
variations = multi_chain.invoke({"question": query})

print(f"Original: {query}")
print("\nGenerated variations:")
print(variations)

In [ ]:
# Retrieve for each variation and merge
all_queries = [query] + [line.strip().lstrip("123.").strip() for line in variations.strip().split("\n") if line.strip()]

all_results = []
seen_content = set()

for q in all_queries:
    results = vectorstore.similarity_search(q, k=3)
    for doc in results:
        # Deduplicate by content
        content_hash = doc.page_content[:100]
        if content_hash not in seen_content:
            seen_content.add(content_hash)
            all_results.append(doc)

print(f"Queries used: {len(all_queries)}")
print(f"Total unique results: {len(all_results)}")
print("\nMerged results:")
for i, doc in enumerate(all_results[:5], 1):
    print(f"  [{i}] {doc.page_content[:120]}...")

## Technique 3: Re-Ranking with Cross-Encoders

**Problem:** Bi-encoder retrieval (what we've been doing) is fast but imprecise. It embeds query and document independently, so it can miss nuanced relevance.

**Solution:** Two-stage retrieval:
1. **Bi-encoder** (fast): retrieve top-20 candidates
2. **Cross-encoder** (accurate): re-score all 20, keep top-5

Cross-encoders process the query-document pair *together*, making them more accurate but too slow for searching the entire collection.

```
Query → Bi-encoder retrieves top-20 → Cross-encoder re-ranks → Top-5 final results
```

In [ ]:
from sentence_transformers import CrossEncoder

# Load a cross-encoder model (small, runs locally)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

query = "What techniques improve retrieval quality in RAG?"

# Stage 1: Retrieve a larger candidate set
candidates = vectorstore.similarity_search(query, k=15)
print(f"Stage 1: Retrieved {len(candidates)} candidates")

# Stage 2: Re-rank with cross-encoder
pairs = [[query, doc.page_content] for doc in candidates]
scores = cross_encoder.predict(pairs)

# Sort by cross-encoder score (descending)
scored_docs = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)

print("\n=== Before Re-ranking (top 5 by bi-encoder) ===")
for i, doc in enumerate(candidates[:5], 1):
    print(f"  [{i}] {doc.page_content[:100]}...")

print("\n=== After Re-ranking (top 5 by cross-encoder) ===")
for i, (score, doc) in enumerate(scored_docs[:5], 1):
    print(f"  [{i}] (score: {score:.4f}) {doc.page_content[:100]}...")

## Comparison: When to Use Each Technique

| Technique | What it improves | Cost | Complexity |
|-----------|-----------------|------|------------|
| **HyDE** | Query-document style mismatch | +1 LLM call | Low |
| **Multi-Query** | Query coverage (captures multiple aspects) | +1 LLM call, +N retrievals | Medium |
| **Re-Ranking** | Retrieval precision (fewer irrelevant docs) | +1 local model inference | Medium |

### Recommendations:
- **Start simple**: Basic similarity or MMR retrieval (notebook 05)
- **If retrieval seems imprecise**: Add re-ranking
- **If queries are short/vague**: Try HyDE
- **If questions are multi-faceted**: Use multi-query
- **For best results**: Combine all three (but measure the improvement!)

## Key Takeaways

1. **Retrieval quality is the bottleneck** — improving retrieval has more impact than switching LLMs
2. **Two-stage retrieval** (bi-encoder + cross-encoder) is a production best practice
3. **Always measure** — use the evaluation framework from notebook 08 to verify improvements
4. **Complexity has a cost** — don't add techniques unless evaluation shows they help

## What's Next?

You've now seen the complete RAG pipeline from document loading to advanced retrieval. Try the [Streamlit app](../app/streamlit_app.py) to interact with the system, or explore the source code in `src/rag_engine/` to understand the implementation details.